# ChronoGrid FusionNet: Hybrid Deep Learning for Transmission Line Fault Classification

**Paper**: *A Hybrid Learning for Detecting and Classifying Transmission Line Faults Using ChronoGrid FusionNet Model*
**Dataset**: WSCC 9-Bus Fault Dataset — 33,000 S-transform heatmap images, 11 fault classes
**Architecture**: 1D CNN → BiLSTM → Self-Attention (ChronoGrid FusionNet)

---

## Abstract

Transmission line faults in the IEEE 9-bus network generate nonlinear transient disturbances in voltage and current waveforms. This notebook implements and evaluates the **ChronoGrid FusionNet** model — a hybrid architecture integrating:

1. **Multi-scale 1D CNN** — localized temporal feature extraction (kernel sizes 3 and 5)
2. **Bidirectional LSTM** — forward and backward temporal dependency modeling
3. **Scaled Dot-Product Self-Attention** — adaptive weighting of critical transient intervals

An **ablation study** is conducted across three configurations to quantify the contribution of each component:

| Model | Architecture |
|-------|-------------|
| Baseline | 1D CNN only |
| Ablation | 1D CNN + BiLSTM |
| Proposed | 1D CNN + BiLSTM + Self-Attention (**ChronoGrid FusionNet**) |

**Temporal interpretation of image data**: Each 227×227 S-transform heatmap encodes frequency content (rows) over time (columns). Treating image rows as frequency-bin channels and columns as time steps provides a natural 1D temporal sequence for the CNN+BiLSTM pipeline.


In [ ]:
import os, sys, warnings, random, time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
import seaborn as sns
from PIL import Image
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_fscore_support,
)
from sklearn.preprocessing import label_binarize
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── IEEE Publication-Quality Style ───────────────────────────────────────────
matplotlib.rcParams.update({
    'font.family'        : 'serif',
    'font.serif'         : ['DejaVu Serif', 'Times New Roman', 'serif'],
    'font.size'          : 10,
    'axes.titlesize'     : 10,
    'axes.titleweight'   : 'bold',
    'axes.labelsize'     : 10,
    'xtick.labelsize'    : 9,
    'ytick.labelsize'    : 9,
    'legend.fontsize'    : 9,
    'legend.framealpha'  : 0.85,
    'figure.dpi'         : 150,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
    'axes.linewidth'     : 0.8,
    'xtick.major.width'  : 0.8,
    'ytick.major.width'  : 0.8,
    'lines.linewidth'    : 1.6,
    'grid.linewidth'     : 0.5,
    'grid.alpha'         : 0.35,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'axes.grid'          : True,
})
sns.set_palette('tab10')

# Colorblind-safe palette for 11 classes
C11 = [
    '#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd',
    '#8c564b','#e377c2','#7f7f7f','#bcbd22','#17becf','#393b79',
]
MODEL_COLORS  = {'Baseline CNN': '#1f77b4', 'CNN+BiLSTM': '#ff7f0e', 'ChronoGrid FusionNet': '#2ca02c'}
MODEL_DASHES  = {'Baseline CNN': (6,0), 'CNN+BiLSTM': (4,2), 'ChronoGrid FusionNet': (1,0)}

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"CUDA — " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('IEEE style configured.')


In [ ]:
# ─── Paths ────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path('/Users/sanathbs/02_College_UG/IEEE-9bus/wscc9FaultDataset')
SAVE_DIR     = DATASET_ROOT          # figures saved here

# ─── Classes (fault-type ordered) ─────────────────────────────────────────────
FAULT_CLASSES = ['AG','BG','CG','AB','AC','BC','ABG','ACG','BCG','ABCG','NF']
NUM_CLASSES   = len(FAULT_CLASSES)
C2I           = {c: i for i, c in enumerate(FAULT_CLASSES)}

# ─── Experiment ───────────────────────────────────────────────────────────────
# Change NUM_SAMPLES_PER_CLASS to 3000 for full training (GPU recommended).
# 300 gives a fast CPU demo in ~10–15 min.
NUM_SAMPLES_PER_CLASS = 300    # <<< set to 3000 for full dataset

IMG_SIZE      = 227
BATCH_SIZE    = 32
EPOCHS        = 30             # <<< set to 50 for full training
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 8
SEED          = 42

# ─── Reproducibility ──────────────────────────────────────────────────────────
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Classes ({NUM_CLASSES}): {FAULT_CLASSES}')
print(f'Samples/class : {NUM_SAMPLES_PER_CLASS}  |  Total : {NUM_SAMPLES_PER_CLASS * NUM_CLASSES:,}')
print(f'Epochs        : {EPOCHS}  |  Batch : {BATCH_SIZE}  |  LR : {LR}')
print(f'Device        : {DEVICE}')


---
## Module 1 — Data Acquisition and Exploratory Analysis


In [ ]:
# ── Count files per class ─────────────────────────────────────────────────────
class_counts = {}
for cls in FAULT_CLASSES:
    cls_dir = DATASET_ROOT / cls
    class_counts[cls] = len(list(cls_dir.glob('*.jpg')))

total = sum(class_counts.values())
print(f'Total images : {total:,}')
print(f'Classes      : {NUM_CLASSES}')
print(f'Balance      : {"Balanced" if len(set(class_counts.values())) == 1 else "Imbalanced"}')
print()
for cls, n in class_counts.items():
    print(f'  {cls:6s} : {n:,} ({n/total*100:.1f}%)')

# ── Figure 1 – Class Distribution ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.0))

counts = [class_counts[c] for c in FAULT_CLASSES]
bars = ax.barh(FAULT_CLASSES, counts, color=C11, edgecolor='white', linewidth=0.5, height=0.65)

for bar, n in zip(bars, counts):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{n:,}', va='center', ha='left', fontsize=9)

ax.set_xlabel('Number of Samples')
ax.set_title('Fig. 1 — WSCC 9-Bus Fault Dataset: Class Distribution', pad=8)
ax.set_xlim(0, max(counts) * 1.15)
ax.tick_params(axis='y', length=0)
ax.set_axisbelow(True)
ax.spines['left'].set_visible(False)

# Fault-type annotation bands
bands = [('Single-phase\nto Ground', [0,1,2]), ('Phase-to-Phase', [3,4,5]),
         ('Two-phase\nto Ground', [6,7,8]), ('Three-phase\nto Ground', [9]), ('No Fault', [10])]
for label, idxs in bands:
    y0 = min(idxs) - 0.42; y1 = max(idxs) + 0.42
    ax.axhspan(y0, y1, alpha=0.06, color=C11[idxs[0]], zorder=0)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig01_class_distribution.png')
plt.show()
print('Fig. 1 saved.')


In [ ]:
# ── Figure 2 – Sample Images per Fault Type ───────────────────────────────────
fig = plt.figure(figsize=(7, 5.6))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.35, wspace=0.08)

fault_labels = {
    'AG':'A-G\n(Single)', 'BG':'B-G\n(Single)', 'CG':'C-G\n(Single)',
    'AB':'A-B\n(Phase)',   'AC':'A-C\n(Phase)',   'BC':'B-C\n(Phase)',
    'ABG':'AB-G\n(DLG)',  'ACG':'AC-G\n(DLG)',   'BCG':'BC-G\n(DLG)',
    'ABCG':'ABC-G\n(3-ph)','NF':'No Fault',
}

for idx, cls in enumerate(FAULT_CLASSES):
    r, c = divmod(idx, 4)
    ax   = fig.add_subplot(gs[r, c])
    img  = Image.open(DATASET_ROOT / cls / '1.jpg')
    ax.imshow(img, aspect='auto', interpolation='lanczos')
    ax.set_title(fault_labels[cls], fontsize=8.5, fontweight='bold', color=C11[idx], pad=3)
    ax.axis('off')

# hide the 12th (empty) subplot
fig.add_subplot(gs[2, 3]).axis('off')

fig.suptitle('Fig. 2 — Representative S-Transform Heatmaps for Each Fault Type',
             fontsize=10, fontweight='bold', y=1.01)

# shared colorbar
sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(0, 1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=fig.axes[:-1], orientation='vertical',
                    fraction=0.015, pad=0.02, shrink=0.85)
cbar.set_label('Normalized Amplitude', fontsize=9)
cbar.ax.tick_params(labelsize=8)

plt.savefig(SAVE_DIR / 'fig02_sample_images.png')
plt.show()
print('Fig. 2 saved.')


In [ ]:
# ── PyTorch Dataset ───────────────────────────────────────────────────────────
class FaultDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.paths  = file_paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('L')          # grayscale
        arr = np.array(img, dtype=np.float32) / 255.0           # [H, W] in [0,1]
        x   = torch.tensor(arr)                                  # [227, 227]
        # Shape interpretation for Conv1d:
        #   dim-0 = 227 frequency bins  → input channels
        #   dim-1 = 227 time steps      → sequence length
        return x, int(self.labels[idx])


# ── Collect file paths ────────────────────────────────────────────────────────
all_paths, all_labels = [], []
for cls in FAULT_CLASSES:
    files = sorted((DATASET_ROOT / cls).glob('*.jpg'))[:NUM_SAMPLES_PER_CLASS]
    all_paths  += [str(f) for f in files]
    all_labels += [C2I[cls]] * len(files)

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)

# ── Stratified split: 70 / 15 / 15 ──────────────────────────────────────────
idx_tr, idx_tmp = train_test_split(
    np.arange(len(all_paths)), test_size=0.30,
    stratify=all_labels, random_state=SEED)
idx_va, idx_te  = train_test_split(
    idx_tmp, test_size=0.50,
    stratify=all_labels[idx_tmp], random_state=SEED)

train_ds = FaultDataset(all_paths[idx_tr], all_labels[idx_tr])
val_ds   = FaultDataset(all_paths[idx_va], all_labels[idx_va])
test_ds  = FaultDataset(all_paths[idx_te], all_labels[idx_te])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train : {len(train_ds):,}  |  Val : {len(val_ds):,}  |  Test : {len(test_ds):,}')
x_sample, y_sample = next(iter(train_loader))
print(f'Batch shape : {tuple(x_sample.shape)}  → Conv1d([B, {IMG_SIZE}_freq, {IMG_SIZE}_time])')


In [ ]:
# ── Figure 3 – Preprocessing Pipeline ────────────────────────────────────────
sample_path = str(list((DATASET_ROOT / 'ABCG').glob('*.jpg'))[0])
rgb_img     = np.array(Image.open(sample_path))
gray_img    = np.array(Image.open(sample_path).convert('L'))
norm_img    = gray_img / 255.0

fig, axes = plt.subplots(1, 4, figsize=(7, 2.1))
titles  = ['(a) RGB Input', '(b) Grayscale', '(c) Normalized [0,1]', '(d) Temporal Sequence']
cmaps   = ['viridis', 'gray', 'inferno', 'viridis']
imgs_to = [rgb_img, gray_img, norm_img, norm_img.T]   # .T shows time on x-axis

for ax, title, img, cmap in zip(axes, titles, imgs_to, cmaps):
    im = ax.imshow(img, cmap=cmap, aspect='auto', interpolation='lanczos')
    ax.set_title(title, fontsize=8.5, pad=4)
    ax.set_xticks([]); ax.set_yticks([])
    ax.spines[:].set_visible(False)

axes[3].set_xlabel('Time steps →', fontsize=8)
axes[3].set_ylabel('Freq. bins ↑', fontsize=8)
axes[3].set_xticks([]); axes[3].set_yticks([])

# Pixel intensity histogram inset
inset = axes[2].inset_axes([0.55, 0.05, 0.42, 0.38])
inset.hist(norm_img.ravel(), bins=40, color='#d62728', alpha=0.75, linewidth=0)
inset.set_xticks([0, 0.5, 1]); inset.set_yticks([])
inset.tick_params(labelsize=6.5)
inset.set_facecolor('#f8f8f8')
for sp in inset.spines.values():
    sp.set_linewidth(0.5)

fig.suptitle('Fig. 3 — Preprocessing Pipeline: ABCG Fault Sample', fontsize=10, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig03_preprocessing.png')
plt.show()
print('Fig. 3 saved.')


---
## Module 2 — Model Development

Three models are implemented to enable a systematic ablation study:

| Model | Description |
|-------|-------------|
| **Baseline CNN** | Multi-scale 1D CNN — captures local temporal patterns only |
| **CNN + BiLSTM** | Adds Bidirectional LSTM — models long-range sequence dependencies |
| **ChronoGrid FusionNet** | Adds Self-Attention — adaptively weights critical transient intervals |

All models share identical CNN feature extractors and classifiers to isolate the contribution of each added component.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Shared CNN backbone (identical across all three models)
# ─────────────────────────────────────────────────────────────────────────────
def make_cnn_backbone():
    return nn.Sequential(
        # Block 1 — kernel-3 (captures short transients)
        nn.Conv1d(IMG_SIZE, 64,  kernel_size=3, padding=1),
        nn.BatchNorm1d(64), nn.ReLU(),
        # Block 2 — kernel-5 (captures broader waveform distortions)
        nn.Conv1d(64,  128, kernel_size=5, padding=2),
        nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),  # → [B, 128, 113]
        # Block 3 — kernel-3 (deep feature abstraction)
        nn.Conv1d(128, 256, kernel_size=3, padding=1),
        nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),  # → [B, 256, 56]
    )

# ─────────────────────────────────────────────────────────────────────────────
# Model A — Baseline: 1D CNN only
# ─────────────────────────────────────────────────────────────────────────────
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.cnn = make_cnn_backbone()
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),          # [B, 256, 1]
            nn.Flatten(),                      # [B, 256]
            nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.head(self.cnn(x))         # x: [B, 227, 227]

# ─────────────────────────────────────────────────────────────────────────────
# Model B — Ablation: 1D CNN + BiLSTM  (no attention)
# ─────────────────────────────────────────────────────────────────────────────
class CNNBiLSTM(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.cnn    = make_cnn_backbone()
        self.bilstm = nn.LSTM(256, 128, num_layers=2, batch_first=True,
                              bidirectional=True, dropout=0.3)
        self.head   = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)     # [B, 56, 256]
        x, _ = self.bilstm(x)                 # [B, 56, 256]
        x = x.mean(dim=1)                     # global avg pool over seq
        return self.head(x)

# ─────────────────────────────────────────────────────────────────────────────
# Model C — Proposed: ChronoGrid FusionNet (1D CNN + BiLSTM + Self-Attention)
# ─────────────────────────────────────────────────────────────────────────────
class _ScaledDotAttn(nn.Module):
    '''Scaled dot-product multi-head self-attention with residual + LayerNorm.'''
    def __init__(self, d_model=256, num_heads=8, dropout=0.1):
        super().__init__()
        self.mha  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout,
                                           batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        out, w = self.mha(x, x, x, need_weights=True, average_attn_weights=True)
        return self.norm(x + out), w          # w: [B, seq, seq]

class ChronoGridFusionNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.cnn       = make_cnn_backbone()
        self.bilstm    = nn.LSTM(256, 128, num_layers=2, batch_first=True,
                                  bidirectional=True, dropout=0.3)
        self.attention = _ScaledDotAttn(d_model=256, num_heads=8)
        self.head      = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x, return_attn=False):
        x = self.cnn(x).permute(0, 2, 1)     # [B, 56, 256]
        x, _ = self.bilstm(x)                 # [B, 56, 256]
        x, attn_w = self.attention(x)         # [B, 56, 256]
        feat = x.mean(dim=1)                  # [B, 256]
        out  = self.head(feat)
        if return_attn:
            return out, attn_w
        return out

# ─── Parameter summary ────────────────────────────────────────────────────────
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

models_info = {
    'Baseline CNN'        : BaselineCNN(),
    'CNN+BiLSTM'          : CNNBiLSTM(),
    'ChronoGrid FusionNet': ChronoGridFusionNet(),
}
hdr = f"{'Model':<25}  {'Parameters':>12}"
print(hdr)
print('-' * 40)
for name, m in models_info.items():
    print(f"{name:<25}  {count_params(m):>12,}")


In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
def train_model(model, train_loader, val_loader, epochs, lr, wd, patience, name='model'):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr*0.05)

    history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    best_val_acc = 0.0
    best_state   = None
    stale        = 0

    pbar = tqdm(range(epochs), desc=f'{name:<25}', ncols=90)
    for epoch in pbar:
        # ── train ──
        model.train()
        tl, tc, tn = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            tl += loss.item() * xb.size(0)
            tc += (out.argmax(1) == yb).sum().item()
            tn += xb.size(0)

        # ── validate ──
        model.eval()
        vl, vc, vn = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out  = model(xb)
                loss = criterion(out, yb)
                vl += loss.item() * xb.size(0)
                vc += (out.argmax(1) == yb).sum().item()
                vn += xb.size(0)

        scheduler.step()

        t_loss = tl/tn; v_loss = vl/vn
        t_acc  = tc/tn; v_acc  = vc/vn

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)

        pbar.set_postfix({'t_acc': f'{t_acc:.3f}', 'v_acc': f'{v_acc:.3f}',
                          'v_loss': f'{v_loss:.4f}'})

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            stale        = 0
        else:
            stale += 1
            if stale >= patience:
                pbar.write(f'  Early stopping at epoch {epoch+1}  (best val_acc={best_val_acc:.4f})')
                break

    model.load_state_dict(best_state)
    model.eval()
    return model, history, best_val_acc

# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    preds, labels, probs = [], [], []
    t0 = time.perf_counter()
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(DEVICE))
            pb  = torch.softmax(out, dim=1)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(yb.numpy())
            probs.extend(pb.cpu().numpy())
    elapsed = time.perf_counter() - t0
    return (np.array(preds), np.array(labels),
            np.array(probs), elapsed / len(loader.dataset) * 1000)   # ms/sample

print('Training utilities ready.')


---
## Training — Ablation Study
Running all three models sequentially. Results are stored in `results` dict for comparison.


In [ ]:
results   = {}   # stores history + eval results per model
model_zoo = {
    'Baseline CNN'        : BaselineCNN(),
    'CNN+BiLSTM'          : CNNBiLSTM(),
    'ChronoGrid FusionNet': ChronoGridFusionNet(),
}

for name, mdl in model_zoo.items():
    print(f'\n{"─"*60}')
    print(f'  Training: {name}')
    print(f'  Params  : {count_params(mdl):,}')
    print(f'  {"─"*58}')
    mdl, hist, best_va = train_model(
        mdl, train_loader, val_loader,
        epochs=EPOCHS, lr=LR, wd=WEIGHT_DECAY,
        patience=PATIENCE, name=name
    )
    preds, lbls, probs, lat = evaluate(mdl, test_loader)
    results[name] = {
        'model'  : mdl,
        'history': hist,
        'preds'  : preds,
        'labels' : lbls,
        'probs'  : probs,
        'latency': lat,
        'best_va': best_va,
        'params' : count_params(mdl),
    }
    acc = (preds == lbls).mean()
    print(f'  Test accuracy : {acc:.4f}  |  Latency : {lat:.3f} ms/sample')

print('\n✓ All models trained and evaluated.')


In [ ]:
# ── Figure 4 — Training Curves (Loss + Accuracy for each model) ──────────────
model_names = list(results.keys())
fig, axes   = plt.subplots(3, 2, figsize=(7, 6.5))
fig.suptitle('Fig. 4 — Training and Validation Curves (Loss & Accuracy)', fontweight='bold', y=1.01)

for row, name in enumerate(model_names):
    hist  = results[name]['history']
    color = MODEL_COLORS[name]
    ep    = range(1, len(hist['train_loss']) + 1)

    # Loss
    ax = axes[row, 0]
    ax.plot(ep, hist['train_loss'], color=color, linewidth=1.6, label='Train')
    ax.plot(ep, hist['val_loss'],   color=color, linewidth=1.6, linestyle='--', label='Val')
    best_ep = int(np.argmin(hist['val_loss'])) + 1
    ax.axvline(best_ep, color='gray', linewidth=0.8, linestyle=':', alpha=0.7)
    ax.scatter([best_ep], [hist['val_loss'][best_ep-1]], s=40, color=color, zorder=5)
    ax.set_ylabel('Cross-Entropy Loss', fontsize=9)
    ax.set_title(f'{name} — Loss', fontsize=9, color=color)
    ax.legend(fontsize=8, loc='upper right')

    # Accuracy
    ax = axes[row, 1]
    ax.plot(ep, [v*100 for v in hist['train_acc']], color=color, linewidth=1.6, label='Train')
    ax.plot(ep, [v*100 for v in hist['val_acc']],   color=color, linewidth=1.6, linestyle='--', label='Val')
    best_ep_a = int(np.argmax(hist['val_acc'])) + 1
    ax.axvline(best_ep_a, color='gray', linewidth=0.8, linestyle=':', alpha=0.7)
    ax.scatter([best_ep_a], [hist['val_acc'][best_ep_a-1]*100], s=40, color=color, zorder=5)
    ax.set_ylabel('Accuracy (%)', fontsize=9)
    ax.set_title(f'{name} — Accuracy', fontsize=9, color=color)
    ax.legend(fontsize=8, loc='lower right')

for ax in axes[-1]:
    ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig04_training_curves.png')
plt.show()
print('Fig. 4 saved.')


In [ ]:
# ── Figure 5 — Ablation: Validation Accuracy of All Three Models ──────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 2.8))

for name in model_names:
    hist  = results[name]['history']
    color = MODEL_COLORS[name]
    dash  = MODEL_DASHES[name]
    ep    = range(1, len(hist['val_loss']) + 1)

    ax1.plot(ep, hist['val_loss'],
             color=color, dashes=dash, linewidth=1.6, label=name)
    ax2.plot(ep, [v*100 for v in hist['val_acc']],
             color=color, dashes=dash, linewidth=1.6, label=name)

ax1.set_xlabel('Epoch'); ax1.set_ylabel('Validation Loss')
ax1.set_title('(a) Validation Loss', fontsize=9, fontweight='bold')
ax1.legend(fontsize=8, loc='upper right')

ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation Accuracy (%)')
ax2.set_title('(b) Validation Accuracy', fontsize=9, fontweight='bold')
ax2.legend(fontsize=8, loc='lower right')

# Annotate final val accuracies
for name in model_names:
    hist  = results[name]['history']
    final = max(hist['val_acc']) * 100
    ep_f  = len(hist['val_acc'])
    ax2.annotate(f'{final:.1f}%',
                 xy=(ep_f, final), xytext=(ep_f - 2, final + 1.5),
                 fontsize=7.5, color=MODEL_COLORS[name],
                 arrowprops=dict(arrowstyle='->', lw=0.8, color=MODEL_COLORS[name]))

fig.suptitle('Fig. 5 — Ablation Study: Progressive Gain from Each Architectural Component',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig05_ablation_curves.png')
plt.show()
print('Fig. 5 saved.')


---
## Module 3 — Performance Evaluation


In [ ]:
# ── Figures 6–8 — Normalised Confusion Matrices ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6))
fig.suptitle('Figs. 6–8 — Normalised Confusion Matrices (Test Set)', fontweight='bold', y=1.02)

short_labels = FAULT_CLASSES

for ax, name in zip(axes, model_names):
    preds = results[name]['preds']
    lbls  = results[name]['labels']
    acc   = (preds == lbls).mean() * 100
    cm    = confusion_matrix(lbls, preds, normalize='true')

    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=short_labels, yticklabels=short_labels,
                ax=ax, linewidths=0.3, linecolor='white',
                cbar_kws={'shrink': 0.75, 'pad': 0.02},
                annot_kws={'size': 6.5})
    ax.set_title(f'{name}\nAcc = {acc:.1f}%', fontsize=9, fontweight='bold',
                 color=MODEL_COLORS[name])
    ax.set_xlabel('Predicted Label', fontsize=9)
    ax.set_ylabel('True Label', fontsize=9)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig06_08_confusion_matrices.png')
plt.show()
print('Figs. 6–8 saved.')


In [ ]:
# ── Figure 9 — Per-Class F1 Score: Ablation Comparison ───────────────────────
f1_data = {}
for name in model_names:
    _, _, f1, _ = precision_recall_fscore_support(
        results[name]['labels'], results[name]['preds'],
        average=None, labels=list(range(NUM_CLASSES))
    )
    f1_data[name] = f1

x     = np.arange(NUM_CLASSES)
width = 0.26
fig, ax = plt.subplots(figsize=(7, 3.0))

for i, name in enumerate(model_names):
    offset = (i - 1) * width
    bars   = ax.bar(x + offset, f1_data[name] * 100, width,
                    color=MODEL_COLORS[name], alpha=0.85, label=name,
                    edgecolor='white', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(FAULT_CLASSES, fontsize=9)
ax.set_ylabel('F1 Score (%)')
ax.set_ylim(0, 108)
ax.set_title('Fig. 9 — Per-Class F1 Score: Ablation Comparison', fontweight='bold')
ax.legend(loc='lower right', fontsize=8.5)
ax.axhline(100, color='gray', linewidth=0.6, linestyle='--', alpha=0.5)

# Annotate final model values on top
for j, cls_f1 in enumerate(f1_data['ChronoGrid FusionNet']):
    ax.text(j + width, cls_f1 * 100 + 1.5, f'{cls_f1*100:.0f}',
            ha='center', va='bottom', fontsize=6.5, color=MODEL_COLORS['ChronoGrid FusionNet'])

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig09_per_class_f1.png')
plt.show()
print('Fig. 9 saved.')


In [ ]:
# ── Figure 10 — Overall Metrics Comparison ────────────────────────────────────
metrics_data = {}
for name in model_names:
    lbls  = results[name]['labels']
    preds = results[name]['preds']
    p, r, f1, _ = precision_recall_fscore_support(lbls, preds, average='macro', zero_division=0)
    acc = (preds == lbls).mean()
    metrics_data[name] = {'Accuracy': acc*100, 'Precision': p*100, 'Recall': r*100, 'F1 (Macro)': f1*100}

metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 (Macro)']
x     = np.arange(len(metric_names))
width = 0.26

fig, ax = plt.subplots(figsize=(5.5, 3.0))
for i, name in enumerate(model_names):
    vals   = [metrics_data[name][m] for m in metric_names]
    offset = (i - 1) * width
    bars   = ax.bar(x + offset, vals, width, label=name,
                    color=MODEL_COLORS[name], alpha=0.88,
                    edgecolor='white', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.4,
                f'{v:.1f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x); ax.set_xticklabels(metric_names, fontsize=9)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 115)
ax.set_title('Fig. 10 — Overall Performance Metrics Comparison', fontweight='bold')
ax.legend(loc='lower right', fontsize=8.5)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig10_overall_metrics.png')
plt.show()
print('Fig. 10 saved.')


In [ ]:
# ── Figure 11 — Multi-Class ROC Curves (ChronoGrid FusionNet) ─────────────────
name   = 'ChronoGrid FusionNet'
lbls   = results[name]['labels']
probs  = results[name]['probs']
y_bin  = label_binarize(lbls, classes=list(range(NUM_CLASSES)))

fpr_all, tpr_all, auc_all = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr_all[i], tpr_all[i], _ = roc_curve(y_bin[:, i], probs[:, i])
    auc_all[i] = auc(fpr_all[i], tpr_all[i])

# Micro-average
fpr_all['micro'], tpr_all['micro'], _ = roc_curve(y_bin.ravel(), probs.ravel())
auc_all['micro'] = auc(fpr_all['micro'], tpr_all['micro'])
# Macro-average
all_fpr = np.unique(np.concatenate([fpr_all[i] for i in range(NUM_CLASSES)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    mean_tpr += np.interp(all_fpr, fpr_all[i], tpr_all[i])
mean_tpr /= NUM_CLASSES
fpr_all['macro'], tpr_all['macro'] = all_fpr, mean_tpr
auc_all['macro'] = auc(all_fpr, mean_tpr)

fig, ax = plt.subplots(figsize=(4.5, 4.0))
# Per-class curves (thin)
for i in range(NUM_CLASSES):
    ax.plot(fpr_all[i], tpr_all[i], color=C11[i], linewidth=0.9, alpha=0.75,
            label=f'{FAULT_CLASSES[i]} (AUC={auc_all[i]:.3f})')
# Aggregate curves (thick)
ax.plot(fpr_all['macro'], tpr_all['macro'], color='#2d2d2d', linewidth=2.0,
        linestyle='--', label=f'Macro avg (AUC={auc_all["macro"]:.3f})')
ax.plot(fpr_all['micro'], tpr_all['micro'], color='#d62728', linewidth=2.0,
        linestyle=':', label=f'Micro avg (AUC={auc_all["micro"]:.3f})')
ax.plot([0,1],[0,1], color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Fig. 11 — Multi-Class ROC Curves\n(ChronoGrid FusionNet)', fontweight='bold')
ax.legend(fontsize=7, loc='lower right', ncol=1)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig11_roc_curves.png')
plt.show()
print('Fig. 11 saved.')


In [ ]:
# ── Figure 12 — Self-Attention Weight Visualization ───────────────────────────
# Show which time-step pairs the model focuses on for representative samples.
chgrid_model = results['ChronoGrid FusionNet']['model'].to(DEVICE)
chgrid_model.eval()

chosen_classes = ['AG', 'AB', 'ABCG', 'NF']
fig, axes = plt.subplots(2, 4, figsize=(7, 4.0))
fig.suptitle('Fig. 12 — Self-Attention Weight Maps and Corresponding S-Transform Images',
             fontweight='bold', y=1.02)

for col, cls in enumerate(chosen_classes):
    cls_idx   = C2I[cls]
    img_path  = list((DATASET_ROOT / cls).glob('*.jpg'))[5]
    pil_img   = Image.open(img_path).convert('L')
    arr       = np.array(pil_img, dtype=np.float32) / 255.0
    x_tensor  = torch.tensor(arr).unsqueeze(0).to(DEVICE)  # [1, 227, 227]

    with torch.no_grad():
        _, attn_w = chgrid_model(x_tensor, return_attn=True)
        # attn_w: [1, seq_len=56, seq_len=56]

    attn_map = attn_w[0].cpu().numpy()   # [56, 56]

    # Row 0: S-transform image
    axes[0, col].imshow(np.array(Image.open(img_path)), aspect='auto',
                        interpolation='lanczos')
    axes[0, col].set_title(f'{cls} Fault', fontsize=9, fontweight='bold',
                            color=C11[FAULT_CLASSES.index(cls)])
    axes[0, col].axis('off')

    # Row 1: Attention weight heatmap
    im = axes[1, col].imshow(attn_map, cmap='hot', aspect='auto',
                              interpolation='nearest', vmin=0)
    axes[1, col].set_title('Attn. Weights', fontsize=8)
    axes[1, col].set_xlabel('Key (time step)', fontsize=7)
    if col == 0:
        axes[1, col].set_ylabel('Query (time step)', fontsize=7)
    axes[1, col].tick_params(labelsize=7)
    plt.colorbar(im, ax=axes[1, col], shrink=0.85, pad=0.02,
                 format='%.2f').ax.tick_params(labelsize=6)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig12_attention_maps.png')
plt.show()
print('Fig. 12 saved.')


---
## Ablation & Comparative Analysis


In [ ]:
# ── Figure 13 — Ablation F1 Heatmap ──────────────────────────────────────────
f1_matrix = np.array([f1_data[n] * 100 for n in model_names])   # [3, 11]
row_labels = ['Baseline\nCNN', 'CNN+\nBiLSTM', 'ChronoGrid\nFusionNet']

fig, ax = plt.subplots(figsize=(7, 2.4))
im = ax.imshow(f1_matrix, aspect='auto', cmap='RdYlGn', vmin=50, vmax=100)

for i in range(len(model_names)):
    for j in range(NUM_CLASSES):
        v = f1_matrix[i, j]
        ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                fontsize=7.5, fontweight='bold',
                color='white' if v < 65 else '#1a1a1a')

ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(FAULT_CLASSES, fontsize=9)
ax.set_yticks(range(len(model_names))); ax.set_yticklabels(row_labels, fontsize=9)
ax.set_xlabel('Fault Class'); ax.set_title('Fig. 13 — F1 Score Heatmap: Ablation Study (Test Set)',
                                            fontweight='bold')
cb = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.03, pad=0.02)
cb.set_label('F1 Score (%)', fontsize=9)
cb.ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig13_ablation_f1_heatmap.png')
plt.show()
print('Fig. 13 saved.')


In [ ]:
# ── Figure 14 — Radar Chart: 5-Metric Ablation Comparison ────────────────────
radar_metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Macro', 'AUC Macro']
N = len(radar_metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]    # close the polygon

radar_vals = {}
for name in model_names:
    lbls  = results[name]['labels']
    preds = results[name]['preds']
    probs = results[name]['probs']
    y_bin_r = label_binarize(lbls, classes=list(range(NUM_CLASSES)))
    p, r, f1, _ = precision_recall_fscore_support(lbls, preds, average='macro', zero_division=0)
    acc = (preds == lbls).mean()
    # macro AUC
    auc_per_cls = []
    for ci in range(NUM_CLASSES):
        fpr_r, tpr_r, _ = roc_curve(y_bin_r[:, ci], probs[:, ci])
        auc_per_cls.append(auc(fpr_r, tpr_r))
    mauc = np.mean(auc_per_cls)
    radar_vals[name] = [acc*100, p*100, r*100, f1*100, mauc*100]

fig, ax = plt.subplots(figsize=(3.8, 3.8), subplot_kw=dict(polar=True))

for name in model_names:
    vals = radar_vals[name] + [radar_vals[name][0]]
    ax.plot(angles, vals, color=MODEL_COLORS[name], linewidth=1.8,
            dashes=MODEL_DASHES[name], label=name)
    ax.fill(angles, vals, color=MODEL_COLORS[name], alpha=0.08)

ax.set_thetagrids(np.degrees(angles[:-1]), radar_metrics, fontsize=9)
ax.set_ylim(50, 102)
ax.set_yticks([60, 70, 80, 90, 100]); ax.set_yticklabels(['60','70','80','90','100'], fontsize=7)
ax.set_rlabel_position(30)
ax.set_title('Fig. 14 — Radar: Multi-Metric\nAblation Comparison', fontweight='bold',
             fontsize=9, pad=18)
ax.legend(loc='lower left', bbox_to_anchor=(-0.22, -0.22), fontsize=8)
ax.grid(True, linewidth=0.6, alpha=0.5)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig14_radar_chart.png')
plt.show()
print('Fig. 14 saved.')


In [ ]:
# ── Table 1 — Comparative Summary ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 1.6))
ax.axis('off')

col_headers = ['Model', 'Params (K)', 'Accuracy (%)', 'Precision (%)', 'Recall (%)',
               'F1 Macro (%)', 'AUC Macro (%)', 'Latency (ms)']
rows_data   = []

for name in model_names:
    lbls  = results[name]['labels']
    preds = results[name]['preds']
    probs = results[name]['probs']
    y_bin_t = label_binarize(lbls, classes=list(range(NUM_CLASSES)))
    p, r, f1, _ = precision_recall_fscore_support(lbls, preds, average='macro', zero_division=0)
    acc = (preds == lbls).mean()
    auc_per_cls = []
    for ci in range(NUM_CLASSES):
        fpr_r, tpr_r, _ = roc_curve(y_bin_t[:, ci], probs[:, ci])
        auc_per_cls.append(auc(fpr_r, tpr_r))
    mauc = np.mean(auc_per_cls)
    params_k = results[name]['params'] / 1000
    lat = results[name]['latency']
    rows_data.append([
        name,
        f'{params_k:.1f}',
        f'{acc*100:.2f}',
        f'{p*100:.2f}',
        f'{r*100:.2f}',
        f'{f1*100:.2f}',
        f'{mauc*100:.2f}',
        f'{lat:.3f}',
    ])

# highlight best values in each numeric column
cols_numeric = list(range(1, len(col_headers)))
col_best = {}
for ci in cols_numeric:
    vals = [float(row[ci]) for row in rows_data]
    # For latency (last col), lower is better
    col_best[ci] = vals.index(min(vals) if ci == len(col_headers)-1 else max(vals))

cell_colors = [['#f0f0f0' if ci == 0 else 'white' for ci in range(len(col_headers))]
               for _ in rows_data]
for ci, best_row in col_best.items():
    cell_colors[best_row][ci] = '#d4edda'   # light green highlight

tbl = ax.table(
    cellText=rows_data,
    colLabels=col_headers,
    cellLoc='center',
    loc='center',
    cellColours=cell_colors,
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.55)

# Style header row
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2d3a4a')
        cell.set_text_props(color='white', fontweight='bold')
    cell.set_edgecolor('#cccccc')
    cell.set_linewidth(0.5)

ax.set_title('Table 1 — Comparative Performance Summary (Test Set, best per column highlighted)',
             fontsize=9, fontweight='bold', pad=6)
plt.tight_layout()
plt.savefig(SAVE_DIR / 'table1_comparative_summary.png')
plt.show()
print('Table 1 saved.')

# Also print classification report for best model
print('\n── Classification Report: ChronoGrid FusionNet ──')
print(classification_report(
    results['ChronoGrid FusionNet']['labels'],
    results['ChronoGrid FusionNet']['preds'],
    target_names=FAULT_CLASSES, digits=4
))


---
## Conclusion

This notebook implements the **ChronoGrid FusionNet** model for WSCC 9-bus transmission line fault classification and quantifies the contribution of each architectural component through an ablation study.

### Key Findings

| Component Added | Contribution |
|----------------|-------------|
| **Baseline 1D CNN** | Captures localized transient patterns from S-transform spectrogram rows |
| **+ BiLSTM** | Models long-range forward/backward temporal dependencies across the time-frequency sequence |
| **+ Self-Attention** | Adaptively weights critical transient intervals; resolves ambiguity between closely related fault signatures |

### Architecture Summary
- **Input representation**: 227×227 grayscale S-transform image → [227 freq-bins, 227 time-steps] sequence
- **1D CNN backbone**: Multi-scale kernels (3, 5) shared across all models
- **BiLSTM**: 2-layer, bidirectional, hidden=128 → output=256
- **Self-Attention**: Scaled dot-product, 8 heads, 256-dim → residual + LayerNorm
- **Optimizer**: Adam with CosineAnnealingLR, gradient clipping (norm=1.0)
- **Loss**: Categorical Cross-Entropy

### Figures Generated
| Figure | Description |
|--------|-------------|
| Fig. 1  | Dataset class distribution |
| Fig. 2  | S-transform sample images (11 fault types) |
| Fig. 3  | Preprocessing pipeline |
| Fig. 4  | Training/validation curves (all 3 models) |
| Fig. 5  | Ablation validation accuracy comparison |
| Figs. 6–8 | Normalised confusion matrices |
| Fig. 9  | Per-class F1 grouped bar chart |
| Fig. 10 | Overall metrics comparison |
| Fig. 11 | Multi-class ROC curves |
| Fig. 12 | Self-attention weight maps |
| Fig. 13 | F1 heatmap (ablation) |
| Fig. 14 | Radar chart (5-metric comparison) |
| Table 1 | Comparative performance summary |

---
*Implemented for: A Hybrid Learning for Detecting and Classifying Transmission Line Faults Using ChronoGrid FusionNet Model*
